# GPY Reader - Compact format for GNSS data

Copyright 2026 Michael George (AKA Logiqx).

This file is part of [GPS Wizard](https://github.com/Logiqx/gps-wizard) and is distributed under the terms of the GNU General Public License.

GPS Wizard is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

GPS Wizard is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with GPS Wizard. If not, see <https://www.gnu.org/licenses/>.

In [1]:
import os
import sys
import time

import numpy as np

import unittest

from base_reader import BaseReader

## Main Class

In [2]:
HEADER_ID = 0xF0
UNCOMPRESSED_RECORD_ID = 0xE0
COMPRESSED_RECORD_ID = 0xD0

class GpyReader(BaseReader):
    '''GPY file - Compact format for GNSS data from the ESP GPS'''

    def __init__(self, filename):
        '''Basic init just records the filename'''

        super().__init__(filename)


    def load(self, ignoreChecksums=False):
        '''Load file into memory'''

        with open(self.filename, "rb") as f:
            self.buffer = f.read()
            
            # The memoryview will be used to access the buffer without intermediate copying
            self.bufferView = memoryview(self.buffer)

            if len(self.buffer) > 0:
                self.readData(ignoreChecksums=ignoreChecksums)


    def readData(self, ignoreChecksums=False):
        '''Read GNSS frames into memory'''

        fileSize = len(self.buffer)

        # Lean buffer will only contain the geodetic navigation data, minus start / end tokens and checksums
        uncompressedBuffer = bytearray()
        compressedBuffer = bytearray()
        recordGroups = []

        numUncompressedRecords = 0
        numCompressedRecords = 0

        while self.bufferPtr < fileSize:
            recordType, payload = self.readFrame(ignoreChecksums=ignoreChecksums)
            
            if recordType == UNCOMPRESSED_RECORD_ID:
                # Start new group of deltas
                deltas = []
                recordGroups.append(deltas)
                
                # Use += because it is faster than extend - see https://stackoverflow.com/q/40004517
                uncompressedBuffer += payload
                numUncompressedRecords += 1
            elif recordType == COMPRESSED_RECORD_ID:
                # Add compressed record to list of deltas
                deltas.append(numCompressedRecords)

                # Use += because it is faster than extend - see https://stackoverflow.com/q/40004517
                compressedBuffer += payload
                numCompressedRecords += 1
                       
        if self.bufferPtr != fileSize:
            raise ValueError('File appears to be truncated')

        # Decode the uncompressed records
        dtype = self.getUncompressedDatatype()
        uncompressedRecords = np.frombuffer(uncompressedBuffer, dtype=dtype, count=numUncompressedRecords)
        #print(uncompressedRecords)

        # Decode the compressed records
        dtype = self.getCompressedDatatype()
        compressedRecords = np.frombuffer(compressedBuffer, dtype=dtype, count=numCompressedRecords)
        #print(compressedRecords)

        # Create data frame for raw data
        self.rawData = np.empty(numUncompressedRecords + numCompressedRecords, dtype=uncompressedRecords.dtype)

        # Combine the records to produce standard raw data
        idx = 0
        for i in range(len(recordGroups)):
            # Copy reference record
            self.rawData[idx] = uncompressedRecords[i]
            idx += 1

            for j in recordGroups[i]:
                # Start by copying the latest reference record
                self.rawData[idx] = uncompressedRecords[i]
                
                # Apply deltas
                self.rawData[idx]['utc'] += compressedRecords[j]['utc_delta']
                self.rawData[idx]['speed'] += compressedRecords[j]['speed_delta']
                self.rawData[idx]['speed_err'] += compressedRecords[j]['speed_err_delta']
                self.rawData[idx]['latitude'] += compressedRecords[j]['latitude_delta']
                self.rawData[idx]['longitude'] += compressedRecords[j]['longitude_delta']
                self.rawData[idx]['course'] += compressedRecords[j]['course_delta'] * 1000

                # Copy regular values
                self.rawData[idx]['hdop'] = compressedRecords[j]['hdop']
                self.rawData[idx]['sats'] = compressedRecords[j]['sats']
                self.rawData[idx]['fix'] = compressedRecords[j]['fix']

                idx += 1

        self.consumeRawData()

        self.summarise()


    def readFrame(self, ignoreChecksums=False):
        '''Read GPY frame from buffer'''

        offset = self.bufferPtr

        leader = self.readBytes(2)
        recordType, flags = leader

        if recordType == UNCOMPRESSED_RECORD_ID:
            length = 36
            payloadSize = length - 4
        elif recordType == COMPRESSED_RECORD_ID:
            length = 20
            payloadSize = length - 4
        else:
            length = self.readUnsigned16LE()
            payloadSize = length - 6

        # Main frame content
        payload = self.readBytes(payloadSize)

        expectedChecksum = self.readUnsigned16LE()

        # Evaluate checksum
        if ignoreChecksums == False:
            # Checksums include type + flags + size (if present), but excludes the actual 2 byte checksum
            record = self.bufferView[offset : offset + length - 2]
            actualChecksum = self.calculateChecksum(record)
            if actualChecksum != expectedChecksum:
                raise ValueError('Checksum error at {} - {} vs {}'.format(offset, actualChecksum, expectedChecksum))

        return recordType, payload


    def calculateChecksum(self, message):
        '''Calculate Fletcher's checksum for the GPY record'''

        checksum_a = checksum_b = 0
        
        # Sum the message bytes
        for byte in message:
            checksum_a += byte
            checksum_b += checksum_a

        # Bit masking has been deferred until the end
        return (checksum_b & 0xff) << 8 | checksum_a & 0xff


    def getUncompressedDatatype(self):
        '''Get the datatype of the uncompressed GNSS frames'''

        # Uncompressed GNSS frame (type = 0xE0) - 32 bytes excluding the type, flags, and checksum
        dtype = [
            ('hdop', '<u2'),
            ('utc', '<u8'),
            ('speed', '<u4'),
            ('speed_err', '<u4'),
            ('latitude', '<i4'),
            ('longitude', '<i4'),
            ('course', '<u4'),
            ('sats', 'u1'),
            ('fix', 'u1')
        ]

        return np.dtype(dtype)


    def getCompressedDatatype(self):
        '''Get the datatype of the compressed GNSS frames'''

        # Compressed GNSS frame (type = 0xD0) - 16 bytes excluding the type, flags, and checksum
        dtype = [
            ('hdop', '<u2'),
            ('utc_delta', '<i2'),
            ('speed_delta', '<i2'),
            ('speed_err_delta', '<i2'),
            ('latitude_delta', '<i2'),
            ('longitude_delta', '<i2'),
            ('course_delta', '<i2'),
            ('sats', 'u1'),
            ('fix', 'u1')
        ]

        return np.dtype(dtype)


    def consumeRawData(self):
        '''Consume data - convert to standard data structure'''

        # Create new track object
        track = self.addTrack()

        # Convert latitude and longitude to decimal
        track.data['lat'] = np.divide(self.rawData['latitude'], 10000000, dtype='float64')
        track.data['lon'] = np.divide(self.rawData['longitude'], 10000000, dtype='float64')

        # Convert SOG from mm/s to m/s
        track.data['sog'] = np.divide(self.rawData['speed'], 1000, dtype='float32')

        # Convert COG from integer to decimal
        track.data['cog'] = np.round(np.divide(self.rawData['course'], 100000, dtype='float32'), 3)

        # Convert timestamp from ms to s
        track.data['ts'] = np.divide(self.rawData['utc'], 1000, dtype='float64')

        # Copy fix and satellites
        track.data['fix'] = self.rawData['fix']
        track.data['sat'] = self.rawData['sats']

        # Convert EHVE (sAcc) from mm/s to m/s
        track.data['ehve'] = np.divide(self.rawData['speed_err'], 1000, dtype='float32')

        # Convert HDOP from integer to decimal
        track.data['hdop'] = np.divide(self.rawData['hdop'], 100, dtype='float32')

## Unit Tests

In [3]:
class TestGpyData(unittest.TestCase):
    '''Class to test GPY data was correctly loaded'''

    def testNumRecords(self):
        '''Test the number of records is as expected'''

        for fieldName in gpyReader.tracks[0].data:
            self.assertEqual(gpyReader.tracks[0].data[fieldName].size, 3395)


    def testHdop(self):
        '''Test the horizontal dilution of precision is as expected'''

        self.assertEqual(gpyReader.tracks[0].data['hdop'].min(), np.float32(0.49))
        self.assertEqual(gpyReader.tracks[0].data['hdop'].max(), np.float32(0.64))


    def testSat(self):
        '''Test the satellite count is as expected'''

        self.assertEqual(gpyReader.tracks[0].data['sat'].min(), np.uint8(22))
        self.assertEqual(gpyReader.tracks[0].data['sat'].max(), np.uint8(32))


    def testTimestamp(self):
        '''Test the timestamp is as expected'''

        self.assertEqual(gpyReader.tracks[0].data['ts'].min(), np.float64(1781024139.6))
        self.assertEqual(gpyReader.tracks[0].data['ts'].max(), np.float64(1781024818.4))


    def testLat(self):
        '''Test the latitude is as expected'''

        self.assertEqual(gpyReader.tracks[0].data['lat'].min(), np.float64(50.5711874))
        self.assertEqual(gpyReader.tracks[0].data['lat'].max(), np.float64(50.5811413))


    def testLon(self):
        '''Test the longitude is as expected'''

        self.assertEqual(gpyReader.tracks[0].data['lon'].min(), np.float64(-2.4684534))
        self.assertEqual(gpyReader.tracks[0].data['lon'].max(), np.float64(-2.4565341))


    def testSog(self):
        '''Test the speed over ground is as expected'''

        self.assertEqual(gpyReader.tracks[0].data['sog'].min(), np.float32(0.006))
        self.assertEqual(gpyReader.tracks[0].data['sog'].max(), np.float32(17.608))


    def testCog(self):
        '''Test the course over ground is as expected'''

        self.assertEqual(gpyReader.tracks[0].data['cog'].min(), np.float32(0.569))
        self.assertEqual(gpyReader.tracks[0].data['cog'].max(), np.float32(357.76))


    def testFix(self):
        '''Test the fix is as expected'''

        self.assertEqual(gpyReader.tracks[0].data['fix'].min(), np.uint8(3))
        self.assertEqual(gpyReader.tracks[0].data['fix'].max(), np.uint8(3))


    def testEhve(self):
        '''Test the estimated horizontal velocity error (speed accuracy) is as expected'''

        self.assertEqual(gpyReader.tracks[0].data['ehve'].min(), np.float32(0.05))
        self.assertEqual(gpyReader.tracks[0].data['ehve'].max(), np.float32(0.431))

In [4]:
if __name__ == '__main__':
    projdir = os.path.realpath(os.path.join(sys.path[0], "..", ".."))

    filename = os.path.join(projdir, 'sessions', '20260609', 'M10_LISA_K888_2606091755.gpy')
    gpyReader = GpyReader(filename)

    pc1 = time.perf_counter()
    gpyReader.load()
    pc2 = time.perf_counter()

    print("\nGPY files loaded in in %0.2f seconds" % (pc2 - pc1))


GPY files loaded in in 0.06 seconds


In [5]:
if __name__ == '__main__':
    # Determine whether session is interactive or batch to facilitate unittest.main(..., exit=testExit)
    import __main__ as main
    testExit = hasattr(main, '__file__')

    unittest.main(argv=['first-arg-is-ignored'], exit=testExit)

..........
----------------------------------------------------------------------
Ran 10 tests in 0.007s

OK
